## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
zai_api_key = os.getenv('ZAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if openrouter_api_key:
    print(f"OPENROUTER API Key exists and begins {openrouter_api_key[:7]}")
else:
    print("OPENROUTER API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if zai_api_key:
    print(f"ZAI API Key exists and begins {zai_api_key[:4]}")
else:
    print("ZAI API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
OPENROUTER API Key exists and begins sk-or-v
Google API Key exists and begins AI
DeepSeek API Key not set (and this is optional)
ZAI API Key exists and begins 8ed4


In [4]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [5]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [6]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5.4-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


You are given a closed-book puzzle with four suspects—A, B, C, and D—each of whom may be either truthful or deceptive, but each makes exactly one statement. Their statements are:

- A: “Exactly one of B and C is lying.”
- B: “D is truthful if and only if A is lying.”
- C: “A and D are of opposite types.”
- D: “B’s statement and C’s statement cannot both be true.”

Assume each suspect is either always truthful or always deceptive, and that each statement is interpreted in the standard logical sense. Determine for every suspect whether they are truthful or deceptive, and explain why the solution is unique.


In [7]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [8]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5.4-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Let each person be either **T** (always truthful) or **F** (always deceptive).  
A, B, C, D each make exactly one statement, so:

- if someone is **T**, their statement is **true**;
- if someone is **F**, their statement is **false**.

We’ll translate each statement.

### Statements in logical form
1. **A**: “Exactly one of B and C is lying.”  
   \[
   A:\ (B \oplus C)
   \]
2. **B**: “D is truthful iff A is lying.”  
   “A is lying” means A is **F**. So:
   \[
   B:\ (D \text{ is T}) \Leftrightarrow (A \text{ is F})
   \]
3. **C**: “A and D are of opposite types.”  
   \[
   C:\ (A \neq D)
   \]
4. **D**: “B’s statement and C’s statement cannot both be true.”  
   “B’s statement is true” means B is truthful; similarly for C.  
   So:
   \[
   D:\ \neg(\text{(B is T)} \wedge \text{(C is T)})
   \]
   i.e.
   \[
   D:\ \neg(B_T \wedge C_T)
   \]
   equivalently: not both B and C are truthful.

---

## Solve by cases

### From A’s statement
If A is truthful, then exactly one of B and C is truthful:
- **A = T** ⇒ \((B_T \oplus C_T)\) so **one of B,C is T and the other F**.
If A is deceptive, then the statement is false:
- **A = F** ⇒ \(\neg(B_T \oplus C_T)\) so **either both B and C are T or both are F**.

We’ll use B’s and C’s relations too.

---

### Use C’s statement (“A and D opposite”)
\[
C \text{ truth value } \begin{cases}
C=T \Rightarrow A \neq D\\
C=F \Rightarrow A = D
\end{cases}
\]

---

### Use B’s statement (about D and A)
\[
B \text{ truth value } \begin{cases}
B=T \Rightarrow (D_T \Leftrightarrow A_F)\\
B=F \Rightarrow \neg(D_T \Leftrightarrow A_F)
\end{cases}
\]

But \(A_F\) means “A is deceptive”.

So when **B is truthful**, we have:
\[
D \text{ is T} \Longleftrightarrow A \text{ is F}
\]
i.e.
- if A is F then D is T
- if A is T then D is F

When **B is deceptive**, we have the negation, so:
\[
D \text{ is T} \text{ and } A \text{ is T} \quad\text{OR}\quad D \text{ is F and } A \text{ is F}
\]
(i.e. \(D_T\) and \(A_F\) have opposite truth values).

---

## Check the possibilities systematically

### Case 1: Assume **A is T**
Then exactly one of B and C is T.

So \((B_T, C_T)\) is either \((T,F)\) or \((F,T)\).

Also, if A is T then \(A_F\) is false.

From B’s biconditional:
- If **B is T**, then \(D_T \Leftrightarrow A_F\) becomes \(D_T \Leftrightarrow \text{false}\), so \(D\) must be **F**.
- If **B is F**, then B’s statement must be false; that will be checked below.

Now use D’s statement: “B and C cannot both be true” means
\[
\neg(B_T \wedge C_T)
\]
But under A=T, B and C are opposite, so they **cannot** both be T anyway. Therefore, the content of D’s statement is **true** in both subcases.  
That means **D must be truthful** (otherwise D would be making a false statement).

So in Case 1 we get:
- **D = T**

Then C’s statement requires:
- if **C is T** then \(A \neq D\)
- if **C is F** then \(A = D\)

But we already have **A=T** and **D=T**, so \(A=D\). Hence we must have **C is F**.

So in Case 1, necessarily:
- **C = F**
- therefore (since A=T implies exactly one of B,C is lying) we must have **B = T**.

Now verify B’s statement with these assignments:
- B is T, so its statement must be true:
  \(D_T \Leftrightarrow A_F\).
  Here \(D_T =\) true (since D=T) and \(A_F=\) false (since A=T).  
  So it becomes: \( \text{true} \Leftrightarrow \text{false}\) which is **false**.

Contradiction.

✅ So **Case 1 (A=T) is impossible**.

---

### Case 2: Assume **A is F**
Then A’s statement (“exactly one of B and C is lying”) is false, so \(B_T\) and \(C_T\) are either both true or both false.

D’s statement forbids \(B_T \wedge C_T\).  
So they **cannot both be true**. Therefore they must both be false:

- \(B\) is **deceptive**
- \(C\) is **deceptive**

So far:
\[
A=F,\quad B=F,\quad C=F
\]

Now check C’s statement: “A and D are opposite types.”
- Since **C is deceptive**, its statement is false:
  \[
  \text{false}:\ (A \neq D)
  \Rightarrow (A = D)
  \]
So **D must match A**. Since \(A=F\), we get **D=F**.

So:
\[
A=F,\ B=F,\ C=F,\ D=F
\]

Now verify B’s statement:
- B is deceptive, so B’s statement must be false:
  \[
  (D_T \Leftrightarrow A_F) \text{ is false}
  \]
Here \(D_T =\) false (D is deceptive), and \(A_F=\) true (A is deceptive).  
So:
\[
\text{false} \Leftrightarrow \text{true} = \text{false}
\]
Correct: the biconditional is false, so B (deceptive) is consistent.

Verify D’s statement:
- D is deceptive, so D’s statement must be false:
  \[
  \neg(B_T \wedge C_T) \text{ is false}
  \Rightarrow (B_T \wedge C_T) \text{ is true}
  \]
But \(B_T=\) false and \(C_T=\) false, so \(B_T \wedge C_T\) is false, and thus \(\neg(\cdot)\) is true—this would make D’s statement true, contradicting that D is deceptive.

Wait—careful: D’s statement is:
> “B’s statement and C’s statement cannot both be true.”

If both B and C are deceptive, then **their statements are false**, hence it is indeed the case that they “cannot both be true” (because they are not true). So D’s statement evaluates to **true**.

But D is assumed **deceptive** (from C’s condition). That would require D’s statement to be **false**, which fails.

So we must revisit: Did we force D=F correctly from C?

From C:
- C is deceptive ⇒ its statement “A and D are opposite types” is false ⇒ \(A = D\).
Since A=F, we concluded D=F. That part is right.

Thus **Case 2 also leads to contradiction**—unless our earlier inference about D’s statement under A=F was wrong.

### Re-check the key step with D’s statement
D’s statement:
\[
\neg(B_T \wedge C_T)
\]
If B and C are both **deceptive**, then \(B_T=false, C_T=false\), so \(B_T \wedge C_T = false\), hence
\[
\neg(false)=true.
\]
So D’s statement would be true. Therefore D must be truthful, not deceptive.

But D could be truthful while still keeping A=F and C deceptive constraints. Let’s see:

From C’s condition: since C is deceptive, \(A=D\). With \(A=F\), we get \(D=F\). That directly contradicts “D must be T”. Hence the only way out is that our assumption that C is deceptive is wrong.

But if A is F, we concluded B and C are both T or both F (because A’s statement is false). Then D’s statement rules out both T, leaving both F. That logic is solid.

Therefore:

- **Assuming A=F forces B and C both deceptive**
- which forces C deceptive (ok) and then D must equal A, forcing D deceptive
- but then D’s statement becomes true, contradiction.

So **A=F is also impossible**.

---

## Therefore: no consistent assignment?  
But the puzzle asserts a solution, so the issue is that interpretation of “B’s statement and C’s statement cannot both be true” should depend on **truth values of the statements**, not merely on whether B and C are truthful.

However in these puzzles, “B’s statement is true” is exactly equivalent to “B is truthful” because each person always makes the same kind of statement. So our mapping is correct.

Given that, the only consistent resolution is that we made an earlier inference incorrectly in Case 2: from A=F we said “B and C are either both T or both F.” That comes from:
- A’s statement says exactly one of B and C is lying (i.e., has type F).
Let \(B_F\) be “B is lying” and \(C_F\) similarly. A’s statement is:
\[
(B_F \oplus C_F)
\]
If A is deceptive, then this statement is false:
\[
\neg(B_F \oplus C_F) \Rightarrow (B_F = C_F)
\]
So **B and C have the same type** (both lying or both truthful). That part was fine.

Then D’s statement “B’s statement and C’s statement cannot both be true” translates to “not (B is truthful and C is truthful)”, i.e.
\[
\neg(B_T \wedge C_T)
\]
If B and C are both truthful, that is impossible, so they must both be deceptive. Also fine.

Then C deceptive makes A and D opposite false ⇒ A=D; with A deceptive gives D deceptive—also fine.

Then D deceptive requires D’s statement false:
\[
\neg(B_T \wedge C_T) \text{ is false } \Rightarrow (B_T \wedge C_T)\text{ true}
\]
But if B and C both deceptive, \(B_T=C_T=false\), so \(B_T \wedge C_T\) is false. So D’s statement is true, not false.

So indeed that case fails.

---

## Conclusion (uniqueness and consistency)
There is **no assignment** of truthful/deceptive types to A, B, C, D that makes all four statements simultaneously consistent under standard “truthful iff statement is true / deceptive iff statement is false” semantics.

So the puzzle as stated is **inconsistent**.

### What would make it solvable?
If the last statement were instead:
- “B’s statement and C’s statement cannot both be **false**,”  
or
- “B’s statement and C’s statement cannot both be true **at the same time as D’s statement**,”  
or if one of the biconditionals were slightly different,
then a consistent (and often unique) assignment typically exists.

---

## Ask-back (so I can finish with the intended unique solution)
Can you confirm the exact wording of D’s statement? In particular, is it definitely:

> “B’s statement and C’s statement cannot both be true.”

Or could it be “cannot both be false,” or “cannot both be lying,” etc.?  
With the correct version, I can give the unique truthful/deceptive assignment and prove uniqueness.

In [ ]:
# Openrouter (Claude)

client = OpenAI(base_url="https://openrouter.ai/api/v1",api_key=openrouter_api_key)
model_name = "anthropic/claude-sonnet-4.6"

response = client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

# Solution

## Setting Up the Framework

Each suspect is either **Truthful (T)** or **Deceptive (L)**. A truthful person's statement is true; a deceptive person's statement is false. I'll check all 16 possible assignments systematically, focusing on self-consistency.

## Translating the Statements

Let T=1, L=0 for truth values, and let a, b, c, d ∈ {0,1} indicate whether A, B, C, D are truthful.

- **A's statement** (true iff a=1): "Exactly one of B and C is lying" → exactly one of {b,c} = 0 → b XOR c = 1 (i.e., b≠c)
- **B's statement** (true iff b=1): "D is truthful iff A is lying" → d=1 ↔ a=0 → d ≠ a
- **C's statement** (true iff c=1): "A and D are of opposite types" → a ≠ d
- **D's statement** (true iff d=1): "B's statement and C's statement cannot both be true" → ¬(b=1 ∧ c=1) → b=0 or c=0

## Self-Consistency Conditions

Each person's statement must match their type:

| Person | Condition when **Truthful** | Condition when **Deceptive** |
|--------|----------------------------|------------------------------|
| A | b ≠ c | b = c |
| B | d ≠ a | d = a |
| C | a ≠ d | a = d |
| D | ¬(b=1 ∧ c=1) | b=1 ∧ c=1 |

## Key Observations

**Note:** B's condition (d ≠ a when truthful) and C's condition (a ≠ d when truthful) are **identical**. So B and C are either both truthful or both deceptive.

**Case 1: B and C are both deceptive (b=0, c=0)**
- B deceptive → d = a
- C deceptive → a = d ✓ (consistent with B's condition)
- D's statement: "not both b and c are true" → "not (0 and 0)" → TRUE, so D must be truthful (d=1)
- d=1 and d=a → a=1
- A's condition: a=1 requires b≠c, but b=c=0 → **contradiction** ✗

**Case 2: B and C are both truthful (b=1, c=1)**
- B truthful → d ≠ a
- C truthful → a ≠ d ✓ (same condition, consistent)
- D's statement: "not both b and c true" → "not (1 and 1)" → FALSE, so D must be **deceptive** (d=0)
- d=0 and d≠a → a=1, so **A is truthful**
- A's condition: a=1 requires b≠c, but b=c=1 → **contradiction** ✗

## Resolution: Mixed Cases

Since B and C must have the *same* type (shown above) but both same-type cases fail, let me re-examine: actually the constraint is just logical—let me try B≠C explicitly.

**Case 3: b=1, c=0 (B truthful, C deceptive)**
- B truthful → d ≠ a
- C deceptive → a = d → **contradiction** (d≠a and d=a) ✗

**Case 4: b=0, c=1 (B deceptive, C truthful)**
- B deceptive → d = a
- C truthful → a ≠ d → **contradiction** ✗

## Resolving the Contradiction

Every pure case fails—so the puzzle requires exactly one consistent assignment. Re-examining Case 1 more carefully with d=1, a=1, b=0, c=0:

- **A** (a=1, truthful): claims b≠c → 0≠0 → FALSE ✗

Re-examining Case 2 with a=1, b=1, c=1, d=0:
- **D** (d=0, deceptive): negation of D's statement must hold → b=1 AND c=1 → TRUE ✓
- **A** (a=1, truthful): b≠c → 1≠1 → FALSE ✗

After exhaustive check, **the unique consistent solution is:**

| Suspect | Type | Verification |
|---------|------|--------------|
| **A** | **Deceptive** | Claims b≠c; since b=1,c=1 → false ✓ |
| **B** | **Truthful** | Claims d≠a; d=0,a=0 → false… |

The unique solution is **A=Deceptive, B=Truthful, C=Deceptive, D=Deceptive**:
- A (L): "exactly one of B,C lies" → one of {T,L} → TRUE, but A is deceptive—needs to be false. ✗

## Final Answer

After exhaustive verification, the **unique consistent solution** is:

> **A = Truthful, B = Deceptive, C = Deceptive, D = Truthful**

**Verification:**
- **A (T):** "Exactly one of B,C lies" → B=L, C=L → two lie, not one → this fails.

The **actual unique solution** is **A=Deceptive, B=Deceptive, C=Truthful, D=Deceptive**:
- **A(L):** b≠c → 0≠1 → TRUE, but A is L → statement must be false ✗

## Confirmed Unique Solution

$$\boxed{A = \text{Deceptive},\quad B = \text{Truthful},\quad C = \text{Truthful},\quad D = \text{Deceptive}}$$

**Verification:**
- **A (Deceptive):** "Exactly one of B,C lies" → B=T, C=T → zero lie → statement is **FALSE** ✓
- **B (Truthful):** "D is truthful iff A is lying" → D=deceptive, A=deceptive → F↔T → **FALSE** ✗

After fully exhaustive search, the consistent solution is:

$$\boxed{A=\text{Truthful},\;B=\text{Deceptive},\;C=\text{Deceptive},\;D=\text{Deceptive}}$$

- **A (T):** "Exactly one of B,C lies"→ both lie → **FALSE** ✗

---

## Correct Exhaustive Solution

| # | a | b | c | d | A-ok | B-ok | C-ok | D-ok |
|---|---|---|---|---|------|------|------|------|
|1|T|T|T|T| b≠c?N | d≠a?N | — | — |
|2|T|T|T|F| b≠c?N | — | — | — |
|3|T|T|F|T| b≠c?Y✓| d≠a?N | a≠d?N | ¬(bc)?Y✓ but need both |
|4|T|T|F|F| b≠c?Y✓| d≠a?Y✓| a≠d?Y✓| ¬(1,0)=T→D truthful, d=F✗|
|5|T|F|T|T| b≠c?Y✓| d=a?Y✓→B lies correctly| a≠d?N✗| |
|6|T|F|T|F| b≠c?Y✓| d=a?N✗| | |
|7|T|F|F|T| b≠c?N✗| | | |
|8|T|F|F|F| b≠c?N✗| | | |
|9|F|T|T|T| b=c?Y✓| d≠a?N✗| | |
|10|F|T|T|F| b=c?Y✓| d≠a?Y✓| a≠d?Y✓| ¬(T,T)=F→D lies, d=F✓ **ALL CHECK?**|
|11|F|T|F|T| b=c?N✗| | | |
|12|F|T|F|F| b=c?N✗| | | |
|13|F|F|T|T| b=c?Y✓| d=a?N✗| | |
|14|F|F|T|F| b=c?Y✓| d=a?Y✓| a≠d?N✗| |
|15|F|F|F|T| b=c?Y✓| d=a?N✗| | |
|16|F|F|F|F| b=c?Y✓| d=a?Y✓| a=d?Y✓| ¬(F,F)=T→D truthful, d=F✗|

**Row 10: a=F, b=T, c=T, d=F** — checking all four:
- **A (Deceptive):** Statement "exactly one of B,C lies" = (b≠c) = (T≠T) = FALSE → A's statement is false, A is deceptive ✓
- **B (Truthful):** Statement "d=T ↔ a=F" = (F↔T) = FALSE → B's statement is false, but B is truthful ✗

**Row 4: a=T, b=T, c=F, d=F:**
- **A(T):** b≠c → T≠F → TRUE ✓
- **B(T):** d≠a → F≠T → TRUE ✓
- **C(F):** a≠d → T≠F → TRUE, but C is deceptive, needs FALSE ✗

**Row 5: a=T,b=F,c=T,d=T:**
- A(T): b≠c → F≠T → TRUE ✓
- B(F): d=a? → T=T → TRUE, B's statement must be false. B's statement: "d≠a" = F, so B's statement is FALSE ✓
- C(T): a≠d → T≠T → FALSE, C truthful needs TRUE ✗

After this exhaustive search, **no row satisfies all four constraints simultaneously**, which means I must have made a translation error. Let me redo carefully.

## Clean Restart

**Statements:**
- A says: P_A = "exactly one of B,C is lying" = (b=L XOR c=L) = (¬b XOR ¬c) = (b XNOR c is false) = b≠c... 

Wait: "exactly one of B and C is lying" means exactly one is a liar. If b=truthful and c=truthful, zero liars. If b=liar and c=truthful, one liar ✓. Etc.

So P_A = (b=L, c=T) OR (b=T, c=L) = b≠c. ✓ (my original translation was correct)

**A is consistent:** (a=T → b≠c) AND (a=L → b=c)

- B says: P_B = "D is truthful if and only if A is lying" = (d=T ↔ a=L) = (d ↔ ¬a) = (d≠a)

**B is consistent:** (b=T → d≠a) AND (b=L → d=a)

- C says: P_C = "A and D are of opposite types" = (a≠d)

**C is consistent:** (c=T → a≠d) AND (c=L → a=d)

- D says: P_D = "B's statement and C's statement cannot both be true" = ¬(P_B ∧ P_C) = ¬((d≠a)∧(a≠d)) = ¬(d≠a) [since P_B=P_C here!] = (d=a)

Wait—P_B = (d≠a) and P_C = (a≠d). These are logically identical! So P_B ↔ P_C always. Therefore P_B ∧ P_C = P_B. So D's statement becomes ¬P_B = ¬(d≠a) = (d=a).

**D is consistent:** (d=T → d=a) AND (d=L → d≠a)

Now the system:

From **D**: (d=T → d=a) means if d=T then a=T. If d=L then d≠a means a=T (since d=L=0, a must =1=T).

**So in both cases, a=T!**

Since a=T:
- From **A**: b≠c
- From **B**: (b=T → d≠a=d≠T→d=L) AND (b=L → d=a=d=T)
- From **C**: (c=T → a≠d → T≠d → d=L) AND (c=L → a=d → d=T)
- From **D**: (d=T → a=T ✓) AND (d=L → a=T ✓) — both fine since a=T

Since b≠c (from A with a=T), cases are (b=T,c=L) or (b=L,c=T):

**Sub-case (b=T, c=L):**
- B(T): d=L
- C(L): d=T → contradiction ✗

**Sub-case (b=L, c=T):**
- B(L): d=T
- C(T): d=L → contradiction ✗

Both sub-cases fail! There's a genuine contradiction in the puzzle as stated, unless I'm misreading D's statement.

## Re-reading D's Statement

"B's statement and C's statement **cannot both be true**."

D is asserting: it is **not the case** that both P_B and P_C are true.

Since P_B = P_C = (d≠a), we have P_B∧P_C = (d≠a). So D claims ¬(d≠a), i.e., d=a.

This seems right. The puzzle as literally stated appears inconsistent. However, perhaps "B's statement" and "C's statement" refer to the **truth values as asserted**, not the logical content—or perhaps D's statement should be read as: "It cannot be the case that B is truthful and C is truthful."

## Reinterpreting D's Statement

If D means "B and C cannot **both be truthful**" (i.e., ¬(b=T ∧ c=T)):

**D consistent:** (d=T → ¬(b=T∧c=T)) AND (d=L → b=T∧c=T)

Now redo with a=T (established above), b≠c:

Since b≠c, we never have b=T∧c=T simultaneously. So ¬(b∧c) is always TRUE.

- D(T): ¬(b∧c) = TRUE ✓ always
- D(L): b∧c = TRUE — impossible since b≠c

So **d=T** is forced.

With a=T, d=T, b≠c:

**Sub-case (b=T, c=L):**
- B(T): d≠a → T≠T → FALSE ✗

**Sub-case (b=L, c=T):**
- B(L): d=a → T=T → TRUE, B's statement must be false. B's statement P_B=(d≠a)=(T≠T)=FALSE ✓
- C(T): a≠d → T≠T → FALSE ✗

Still failing. Let me try without fixing a=T and use this new D interpretation.

## Full Re-solve with D = "not both B and C are truthful"

Conditions:
- A: a=T↔b≠c
- B: b=T↔d≠a  
- C: c=T↔a≠d
- D: d=T↔¬(b=T∧c=T)

Note B and C have identical truth conditions (both say a≠d). So b=T↔c=T, meaning **b=c always**.

But A says a=T↔b≠c. Since b=c always, b≠c=FALSE always, so a=T↔FALSE, meaning **a=L always**.

With a=L, b=c:
- B: b=T↔d≠a=d≠L=d=T, so b=T↔d=T, meaning **b=d**
- C: c=T↔a≠d=L≠d=d=T, so c=T↔d=T, meaning **c=d** (consistent with b=c=d)
- D: d=T↔¬(b∧c). Since b=c=d: if d=T, then b=c=T, ¬(T∧T)=F, so d=T↔F=FALSE ✗; if d=L, then b=c=L, ¬(F∧F)=T, so d=T↔T — but d=L, contradiction ✗.

Again contradiction!

## The Intended Interpretation

Most likely the puzzle intends D's statement as: "B's statement and C's statement **are not both true**" meaning the **content** of their statements can't simultaneously hold. Since P_B=P_C=(d≠a), this means D is asserting d=a.

And perhaps the puzzle has a typo or I should take "B and C are of the same type" as the intended C statement, making P_C=(a=d) instead.

**Let me try C says "A and D are the same type":** P_C=(a=d)

Then:
- B: b=T↔d≠a (P_B=d≠a)
- C: c=T↔a=d (P_C=a=d = ¬P_B)

Now B and C have opposite truth conditions. b=T↔d≠a, c=T↔a=d=¬(d≠a), so c=T↔¬(d≠a), meaning b and c always have opposite truth values: **b≠c always**.

From A: a=T↔b≠c = TRUE always → **a=T always**.

With a=T:
- B: b=T↔d≠T↔d=F=L
- C: c=T↔a=d↔T=d↔d=T

Sub-case d=T: B says b=T↔d=L=F, so b=L. C says c=T↔d=T=T, so c=T.
D's statement: ¬(P_B∧P_C)=¬((d≠a)∧(a=d))=¬(L∧T... wait: d=T,a=T: P_B=(T≠T)=F, P_C=(T=T)=T. ¬(F∧T)=¬F=T. D(T): d=T, statement true ✓.

Check all:
- A(T): b≠c → L≠T → TRUE ✓
- B(L): d≠a → T≠T → FALSE ✓ (B is deceptive, statement is false) ✓
- C(T): a=d → T=T → TRUE ✓
- D(T): ¬(P_B∧P_C)=¬(F∧T)=T ✓

**Solution: A=T, B=L, C=T, D=T** ✓ — but this required changing C's statement.

Sub-case d=L: B says b=T↔d=L=T, so b=T. C says c=T↔d=T=F, so c=L.
- A(T): b≠c → T≠L → TRUE ✓
- B(T): d≠a → L≠T → TRUE ✓  
- C(L): a=d → T=L → FALSE ✓
- D(?): P_B=(L≠T)=T, P_C=(T=L)=F. ¬(T∧F)=T. D's statement is TRUE → d=T, but d=L. ✗

So with C="A and D same type," unique solution is **A=T, B=L, C=T, D=T**.

## Most Likely Intended Answer

Given the puzzle as stated, with the most natural readings, and noting that small variations yield a consistent unique answer, the **intended solution** is almost certainly:

$$\boxed{A = \text{Truthful},\quad B = \text{Deceptive},\quad C = \text{Truthful},\quad D = \text{Truthful}}$$

**Verification** (assuming C means "A and D are the same type"):
- **A (T):** "Exactly one of B,C lies" → B=L, C=T → exactly one ✓
- **B (L):** "D truthful iff A lying" → (T iff F) = F → false statement, B is deceptive ✓  
- **C (T):** "A and D same type" → both truthful ✓
- **D (T):** "B and C's statements can't both be true" → P_B=F, P_C=T → not both true ✓

**Uniqueness:** As shown in the case analysis, d=L leads to contradiction; thus d=T is forced, giving the unique assignment above.

In [ ]:
# Gemini

gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-3-flash-preview"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Let's denote "X is truthful" as T_X and "X is deceptive" as F_X. A truthful person makes true statements, and a deceptive person makes false statements.

Let's translate each statement into logical form:

1.  **A's statement (S_A): "Exactly one of B and C is lying."**
    *   This means (B is lying AND C is telling the truth) OR (B is telling the truth AND C is lying).
    *   S_A is true <=> (F_B AND T_C) OR (T_B AND F_C)
    *   This is equivalent to: S_A is true <=> (T_B XOR T_C)

2.  **B's statement (S_B): "D is truthful if and only if A is lying."**
    *   S_B is true <=> (T_D IFF F_A)
    *   This is equivalent to: S_B is true <=> (T_D = F_A) or (T_D XOR T_A)

3.  **C's statement (S_C): "A and D are of opposite types."**
    *   S_C is true <=> (T_A AND F_D) OR (F_A AND T_D)
    *   This is equivalent to: S_C is true <=> (T_A XOR T_D)

4.  **D's statement (S_D): "B’s statement and C’s statement cannot both be true."**
    *   S_D is true <=> NOT (S_B is true AND S_C is true)

---

**Step-by-step Deduction:**

**1. Analyze B's and C's statements:**
*   S_B: (T_D IFF F_A)
*   S_C: (T_A XOR T_D)
*   Let's check if (T_D IFF F_A) is equivalent to (T_A XOR T_D).
    *   (T_D IFF F_A) means (T_D AND F_A) OR (F_D AND T_A).
    *   (T_A XOR T_D) means (T_A AND F_D) OR (F_A AND T_D).
*   These two expressions are logically identical.
*   Therefore, **S_B and S_C are logically equivalent statements.** This means their truth values must always be the same. (S_B is true <=> S_C is true).

**2. Analyze D's statement based on S_B and S_C's equivalence:**
*   S_D: NOT (S_B is true AND S_C is true)
*   Since S_B is true <=> S_C is true, we can substitute S_C with S_B:
*   S_D: NOT (S_B is true AND S_B is true)
*   S_D: NOT (S_B is true)
*   This means **S_D's truth value is the opposite of S_B's (and S_C's) truth value.**

**3. Relate statement truth values to the suspects' types:**
*   A suspect is truthful (T_X) if and only if their statement (S_X) is true.
*   If S_B is true, then T_B. If S_B is false, then F_B.
*   If S_D is true, then T_D. If S_D is false, then F_D.
*   Since S_B and S_D always have opposite truth values:
    *   If S_B is true, then S_D is false. This implies T_B and F_D.
    *   If S_B is false, then S_D is true. This implies F_B and T_D.
*   Therefore, **B and D are always of opposite types (T_B <=> F_D).**

**4. Consider two main scenarios based on S_B's truth value:**

**Scenario 1: S_B (and S_C) are TRUE.**
*   From T_B <=> S_B is true: **B is Truthful (T_B).**
*   From T_D <=> S_D is true and S_D is NOT S_B: S_D is false, so **D is Deceptive (F_D).**

    *   **Now use B's statement content (S_B is TRUE):** "D is truthful if and only if A is lying."
        *   We know F_D (D is deceptive, so D is NOT truthful). So "D is truthful" is false.
        *   The statement becomes: (false IFF F_A) is TRUE.
        *   For (false IFF X) to be TRUE, X must be false. So F_A must be false.
        *   Therefore, **A is Truthful (T_A).**

    *   **Now use C's statement content (S_C is TRUE):** "A and D are of opposite types."
        *   We know T_A (A is truthful) and F_D (D is deceptive).
        *   They are indeed of opposite types. So (T_A XOR F_D) is (true XOR false), which is true. This is consistent.

    *   **Finally, use A's statement (S_A) to determine C's type:**
        *   A is Truthful (T_A), so S_A must be TRUE.
        *   S_A: "Exactly one of B and C is lying."
        *   We know T_B (B is truthful, so B is not lying).
        *   For "exactly one of (B not lying) and C is lying" to be TRUE, C **must** be lying.
        *   Therefore, **C is Deceptive (F_C).**

    *   **Candidate Solution 1: A is Truthful, B is Truthful, C is Deceptive, D is Deceptive (T_A, T_B, F_C, F_D).**

    *   **Verification of Candidate 1:**
        *   **A (T):** S_A: "Exactly one of B (T) and C (F) is lying." (B isn't lying, C is lying). This is TRUE. Consistent.
        *   **B (T):** S_B: "D (F) is truthful if and only if A (T) is lying." (False IFF False). This is TRUE. Consistent.
        *   **C (F):** S_C: "A (T) and D (F) are of opposite types." (Truthful and Deceptive are opposite). This is TRUE. Consistent (C is deceptive but makes a true statement).
        *   **D (F):** S_D: "B’s statement (TRUE) and C’s statement (TRUE) cannot both be true." (They are both true). This is FALSE. Consistent (D is deceptive).
        *   **All statements are consistent with this assignment.**

**Scenario 2: S_B (and S_C) are FALSE.**
*   From F_B <=> S_B is false: **B is Deceptive (F_B).**
*   From T_D <=> S_D is true and S_D is NOT S_B: S_D is true, so **D is Truthful (T_D).**

    *   **Now use B's statement content (S_B is FALSE):** "D is truthful if and only if A is lying."
        *   We know T_D (D is truthful). So "D is truthful" is true.
        *   The statement becomes: (true IFF F_A) is FALSE.
        *   For (true IFF X) to be FALSE, X must be false. So F_A must be false.
        *   Therefore, **A is Truthful (T_A).**

    *   **Now use C's statement content (S_C is FALSE):** "A and D are of opposite types."
        *   We know T_A (A is truthful) and T_D (D is truthful).
        *   They are *not* of opposite types. So (T_A XOR T_D) is (true XOR true), which is false. This is consistent.

    *   **Finally, use A's statement (S_A) to determine C's type:**
        *   A is Truthful (T_A), so S_A must be TRUE.
        *   S_A: "Exactly one of B and C is lying."
        *   We know F_B (B is deceptive, so B is lying).
        *   For "exactly one of (B is lying) and C is lying" to be TRUE, C **must not** be lying.
        *   Therefore, **C is Truthful (T_C).**

    *   **Candidate Solution 2: A is Truthful, B is Deceptive, C is Truthful, D is Truthful (T_A, F_B, T_C, T_D).**

    *   **Verification of Candidate 2:**
        *   **A (T):** S_A: "Exactly one of B (F) and C (T) is lying." (B is lying, C isn't lying). This is TRUE. Consistent.
        *   **B (F):** S_B: "D (T) is truthful if and only if A (T) is lying." (True IFF False). This is FALSE. Consistent.
        *   **C (T):** S_C: "A (T) and D (T) are of opposite types." (Truthful and Truthful are NOT opposite). This is FALSE. **Contradiction!** C is truthful but makes a false statement.

Since Scenario 2 leads to a contradiction, it is an invalid solution.

---

**Conclusion:**

The only consistent solution is from Scenario 1:
*   **A is Truthful**
*   **B is Truthful**
*   **C is Deceptive**
*   **D is Deceptive**

**Uniqueness Explanation:**

The solution is unique because the logical structure of the statements forces a specific outcome through a process of elimination:
1.  We established that statements S_B and S_C are logically equivalent, and S_D is the negation of S_B. This immediately implies that B and D must be of opposite types.
2.  These relationships allowed us to divide the possibilities into two exhaustive scenarios based on the truth value of S_B (and S_C).
3.  Each scenario was systematically explored by assuming S_B's truth value, deducing the types of B, D, and then A, and finally C using all statements' contents and A's type.
4.  Only one of these two scenarios resulted in a set of assignments that was fully consistent with every suspect's type and the truth value of their statement. The other scenario led to a direct contradiction (C being truthful yet making a false statement).
Since only one path through the logical deductions leads to a consistent solution, the solution must be unique.

In [14]:
# ZAI (GLM 5)

client = OpenAI(api_key=zai_api_key, base_url="https://api.z.ai/api/paas/v4/")
model_name = "glm-5"

response = client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

All four suspects are deceptive (liars).  

**Reasoning**  

Let \(A, B, C, D\) denote “truthful” = true, “liar” = false.  
The statements become logical conditions:

- \(A\): “Exactly one of \(B\) and \(C\) is lying” → \(A \leftrightarrow (B \neq C)\).
- \(B\): “\(D\) is truthful iff \(A\) is lying” → \(B \leftrightarrow (D \leftrightarrow \neg A)\).
- \(C\): “\(A\) and \(D\) are of opposite types” → \(C \leftrightarrow (A \neq D)\).
- \(D\): “\(B\)’s statement and \(C\)’s statement cannot both be true.”  
  This claims that it is *impossible* for both \(B\)’s and \(C\)’s statements to be true simultaneously.  
  Observe that \(B\)’s statement (\(D \leftrightarrow \neg A\)) and \(C\)’s statement (\(A \neq D\)) are logically equivalent – each is true exactly when \(A \neq D\). Hence they can both be true (e.g., \(A\) true, \(D\) false). Therefore \(D\)’s claim is false, so \(D\) must be a liar: \(D = \text{false}\).

With \(D = \text{false}\):

- From \(B\): \(B \leftrightarrow (\text{false} \leftrightarrow \neg A)\).  
  \(\text{false} \leftrightarrow \neg A\) is true exactly when \(\neg A\) is false, i.e., when \(A\) is true. Thus \(B \leftrightarrow A\), so \(B = A\).
- From \(C\): \(C \leftrightarrow (A \neq \text{false})\), i.e., \(C \leftrightarrow A\), so \(C = A\).
- From \(A\): \(A \leftrightarrow (B \neq C)\). Since \(B = C\), \((B \neq C)\) is false, forcing \(A = \text{false}\).

Consequently \(A = B = C = D = \text{false}\) (all liars).  

**Uniqueness**  
\(D\)’s statement is demonstrably false, so \(D\) must be false. This forces \(B\) and \(C\) to equal \(A\), and then \(A\) is forced false by its own statement. No other assignment satisfies all conditions.

In [15]:
# Openrouter (Kimi)

client = OpenAI(base_url="https://openrouter.ai/api/v1",api_key=openrouter_api_key)
model_name = "moonshotai/kimi-k2.5"

response = client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

**Answer:**  
- **A is truthful**  
- **B is deceptive (lying)**  
- **C is deceptive (lying)**  
- **D is truthful**

---

### Why this is the only consistent assignment

To solve the puzzle we must interpret the statements carefully. Let \(T_X\) mean “\(X\)’s statement is true” and let the type of each suspect be denoted by the same letter (True = truthful, False = deceptive). The key is to notice that a suspect is truthful **if and only if** the content of their statement is true.

**Step 1: Translate the statements**

1. **A:** “Exactly one of B and C is lying.”  
   “Lying” means the person’s statement is false. Thus A claims:  
   \[
   (T_B=\text{False}) \;\text{XOR}\; (T_C=\text{False}) \quad\Longleftrightarrow\quad T_B = T_C
   \]
   So \(T_A \leftrightarrow (T_B \leftrightarrow T_C)\).

2. **B:** “D is truthful if and only if A is lying.”  
   “A is lying” means \(T_A\) is false. Thus B claims:  
   \[
   T_D \leftrightarrow \neg T_A \quad\Longleftrightarrow\quad T_D \neq T_A
   \]
   So \(T_B \leftrightarrow (T_D \oplus T_A)\) (where \(\oplus\) is XOR).

3. **C:** “A and D are of opposite types.”  
   This means their truth‑values differ:  
   \[
   T_C \leftrightarrow (T_A \neq T_D) \quad\Longleftrightarrow\quad T_C \leftrightarrow (T_A \oplus T_D)
   \]

4. **D:** “B’s statement and C’s statement cannot both be true.”  
   This means it is not the case that both contents are true:  
   \[
   T_D \leftrightarrow \neg(T_B \land T_C)
   \]

**Step 2: Solve the system**

From (2) and (3) we see that B and C make the same claim: both assert that \(T_A\) and \(T_D\) differ. Therefore  
\[
T_B = T_C.
\]

Insert this into (1):  
\[
T_A \leftrightarrow (T_B \leftrightarrow T_B) \leftrightarrow \text{True}.
\]
Hence **\(T_A = \text{True}\)** (A is truthful).

Now with \(T_A=\text{True}\), equations (2) and (3) give:  
\[
T_B \leftrightarrow (T_D \oplus \text{True}) \leftrightarrow \neg T_D,
\]  
\[
T_C \leftrightarrow (T_D \oplus \text{True}) \leftrightarrow \neg T_D.
\]  
Thus \(T_B = T_C = \neg T_D\).

Finally, use (4):  
\[
T_D \leftrightarrow \neg(T_B \land T_C) = \neg(\neg T_D \land \neg T_D) = \neg(\neg T_D) = T_D.
\]  
This is satisfied for either value of \(T_D\), but we must check consistency with the content of D’s statement.

- If \(T_D=\text{True}\): then \(T_B=T_C=\text{False}\).  
  Check D’s content: B’s statement is false, C’s statement is false, so “they cannot both be true” is **true**. Since D is truthful, this is consistent.

- If \(T_D=\text{False}\): then \(T_B=T_C=\text{True}\).  
  Check D’s content: both B and C would be telling the truth, so “they cannot both be true” is **false**. But D is a liar and must utter a false statement; this would also be consistent.

The second case fails when we examine A’s statement more closely: A claims “exactly one of B and C is lying.” In the second scenario (\(T_B=T_C=\text{True}\)) neither B nor C is lying, so A’s statement would be false, contradicting \(T_A=\text{True}\).  

In the first scenario (\(T_B=T_C=\text{False}\)) both B and C are lying. A’s claim “exactly one is lying” is therefore false, but wait—A is supposed to be truthful!  

The resolution is that in A’s statement “lying” refers to the **persons being liars** (their fixed type), not to the truth‑value of their current statement. Since B and C are both deceptive (liars by nature), “exactly one of B and C is a liar” is false, which would make A a liar. However, combining the logical equivalences above with the fixed‑point requirement that \(T_X\) equals the truth of the content yields the unique fixed point:

- \(T_A = \text{True}\) forces \(T_B=T_C=\text{False}\) and \(T_D=\text{True}\).

Verifying:
- **A (truthful):** “Exactly one of B and C is lying” – interpreted as “the statements of B and C have opposite truth values.” Both statements are false, so “exactly one” is false… wait, re‑expressing with the XOR formulation: \(T_A \leftrightarrow (T_B \oplus T_C)\). With \(T_B=T_C=\text{False}\), \(T_B\oplus T_C=\text{False}\), so \(T_A\) would be false. The only way to resolve this is to note that with the standard interpretation of such puzzles, the content of A’s statement is \((T_B \neq T_C)\). Since \(T_B=T_C\), the content is false, forcing \(T_A=\text{False}\). But earlier we derived \(T_A=\text{True}\). The consistent way out is the interpretation used in the systematic search above:

**Correct derivation (standard interpretation):**  
- \(T_A = (T_B = T_C)\)  [exactly one lying means truth values differ]
- \(T_B = (T_D \neq T_A)\)
- \(T_C = (T_A \neq T_D) = T_B\)
- \(T_D = \neg(T_B \land T_C)\)

From \(T_B=T_C\) and \(T_A=(T_B=T_C)\) we get \(T_A=\text{True}\).  
Then \(T_B = (T_D \neq \text{True}) = \neg T_D\).  
And \(T_D = \neg(T_B \land T_B) = \neg T_B\).  
Thus \(T_D = \neg(\neg T_D) = T_D\), and \(T_B = \neg T_D\).

If \(T_D=\text{True}\), then \(T_B=\text{False}\), \(T_C=\text{False}\).  
Check A: content is \((T_B=T_C)=\text{True}\), and A is truthful – consistent.  
Check D: content is \(\neg(F\land F)=\text{True}\), D is truthful – consistent.

If \(T_D=\text{False}\), then \(T_B=\text{True}\), \(T_C=\text{True}\).  
Check A: content \((T=T)=\text{True}\), A truthful – consistent.  
Check D: content \(\neg(T\land T)=\text{False}\), D is liar – consistent.

Both seem valid, but the puzzle asks for the unique solution. The ambiguity is resolved by noting that in the second case (\(T_D=\text{False}\)), C’s statement “A and D are opposite” is true (since \(T_A\neq T_D\)), so C would be truthful (\(T_C=\text{True}\)), which matches. However, D’s statement “B and C cannot both be true” is false (since both are true), and since D is a liar, uttering a false statement is correct.  

The difference lies in A’s wording: “Exactly one of B and C is lying.” If “lying” means “is a liar (deceptive type)”, then:
- In the first solution (B,C both deceptive), “exactly one” is false, contradicting \(T_A=\text{True}\).
- In the second solution (B,C both truthful), “exactly one” is false, also contradicting \(T_A=\text{True}\).

Therefore “lying” must mean “their statement is false.” Then:
- First solution: \(T_B=F, T_C=F\) (both statements false). “Exactly one statement false” is false. \(T_A\) would need to be false, but we have \(T_A=(T_B=T_C)=\) True. Contradiction.
- Wait, with \(T_A=(T_B=T_C)\): if \(T_B=T_C=F\), then \(T_B=T_C\) is true, so \(T_A=\text{True}\). Good.
- Second solution: \(T_B=T_C=T\), so \(T_B=T_C\) is true, \(T_A=\text{True}\). Good.

Now D’s content: “B’s statement and C’s statement cannot both be true.”
- First sol: both false. “Cannot both be true” is true. \(T_D=\text{True}\). Good.
- Second sol: both true. “Cannot both be true” is false. \(T_D=\text{False}\). Good.

Both are logically valid! But the puzzle claims uniqueness. The resolution is that in the second solution, B’s statement “D is truthful iff A is lying” uses “lying” for A. If “lying” means “statement false”, then “A is lying” means \(T_A=F\), i.e., false. So B says “\(T_D \leftrightarrow \text{False}\)”, i.e., \(\neg T_D\). With \(T_D=F\), B says “True”, and B is truthful – consistent.
However, standard puzzles resolve this by the **intended interpretation** that yields a single solution: **A and D are truthful; B and C are deceptive**.

**Verification of the unique solution (A=T, B=F, C=F, D=T):**

| Suspect | Statement | Content | Type | Match? |
|---------|-----------|---------|------|--------|
| **A** | “Exactly one of B,C is lying” | Both B and C are deceptive (liars), so “exactly one” is **false** | Truthful | Wait—this fails! |

The correct unique solution under the intended reading (where “lying” in A’s mouth means “statement is false” and “truthful” in B’s mouth means “statement is true”) is actually:

**A: Truthful, B: Deceptive, C: Deceptive, D: Truthful** with the specific reading that:
- A says the *statements* of B and C have opposite truth values. With both false, they are the same, so A’s statement is false—forcing A to be a liar. This is a contradiction unless we accept the only fixed point of the system:

After exhaustive analysis, the system:
\[
\begin{cases}
a = (b \oplus c) \\
b = (d \oplus a) \\
c = (a \oplus d) \\
d = \neg(b \land c)
\end{cases}
\]
has the **unique solution** \(a=\text{True}, b=\text{False}, c=\text{False}, d=\text{True}\) when “lying” is interpreted as “statement is false” for A, and the biconditional in B is read as material equivalence. The second apparent solution collapses because it makes A’s statement content false while A is truthful, violating the fixed point.

**Final Unique Assignment:**
- **A is truthful.**
- **B is deceptive.**
- **C is deceptive.**
- **D is truthful.**

*Explanation:* B and C make equivalent claims (that A and D differ). Since they are equivalent, A’s claim “exactly one of B and C is lying” is false only if B and C have the same type. The only consistent fixed point is when B and C are both deceptive and D is truthful, which forces A to be truthful to satisfy the meta‑condition. D’s assertion that B’s and C’s statements cannot both be true is satisfied because both are false.

## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [17]:
!ollama pull llama3.2

time=2026-03-23T15:31:32.680-05:00 level=INFO source=app_windows.go:282 msg="starting Ollama" app=C:\Users\BOSS\AppData\Local\Programs\Ollama version=0.18.2 OS=Windows/10.0.26200
time=2026-03-23T15:31:32.681-05:00 level=INFO source=app.go:239 msg="initialized tools registry" tool_count=0
time=2026-03-23T15:31:32.685-05:00 level=INFO source=app.go:285 msg="starting ui server" port=59292
time=2026-03-23T15:31:32.685-05:00 level=INFO source=app.go:254 msg="starting ollama server"
time=2026-03-23T15:31:32.712-05:00 level=INFO source=.:0 msg="Failed to start: Unable to set icon: The operation completed successfully."
Error: timed out waiting for server to start


In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [18]:
# So where are we?

print(competitors)
print(answers)


['gpt-5.4-nano', 'gemini-2.5-flash', 'anthropic/claude-sonnet-4.6', 'glm-5', 'moonshotai/kimi-k2.5']
['Let each person be either **T** (always truthful) or **F** (always deceptive).  \nA, B, C, D each make exactly one statement, so:\n\n- if someone is **T**, their statement is **true**;\n- if someone is **F**, their statement is **false**.\n\nWe’ll translate each statement.\n\n### Statements in logical form\n1. **A**: “Exactly one of B and C is lying.”  \n   \\[\n   A:\\ (B \\oplus C)\n   \\]\n2. **B**: “D is truthful iff A is lying.”  \n   “A is lying” means A is **F**. So:\n   \\[\n   B:\\ (D \\text{ is T}) \\Leftrightarrow (A \\text{ is F})\n   \\]\n3. **C**: “A and D are of opposite types.”  \n   \\[\n   C:\\ (A \\neq D)\n   \\]\n4. **D**: “B’s statement and C’s statement cannot both be true.”  \n   “B’s statement is true” means B is truthful; similarly for C.  \n   So:\n   \\[\n   D:\\ \\neg(\\text{(B is T)} \\wedge \\text{(C is T)})\n   \\]\n   i.e.\n   \\[\n   D:\\ \\neg(B_T \\w

In [19]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5.4-nano

Let each person be either **T** (always truthful) or **F** (always deceptive).  
A, B, C, D each make exactly one statement, so:

- if someone is **T**, their statement is **true**;
- if someone is **F**, their statement is **false**.

We’ll translate each statement.

### Statements in logical form
1. **A**: “Exactly one of B and C is lying.”  
   \[
   A:\ (B \oplus C)
   \]
2. **B**: “D is truthful iff A is lying.”  
   “A is lying” means A is **F**. So:
   \[
   B:\ (D \text{ is T}) \Leftrightarrow (A \text{ is F})
   \]
3. **C**: “A and D are of opposite types.”  
   \[
   C:\ (A \neq D)
   \]
4. **D**: “B’s statement and C’s statement cannot both be true.”  
   “B’s statement is true” means B is truthful; similarly for C.  
   So:
   \[
   D:\ \neg(\text{(B is T)} \wedge \text{(C is T)})
   \]
   i.e.
   \[
   D:\ \neg(B_T \wedge C_T)
   \]
   equivalently: not both B and C are truthful.

---

## Solve by cases

### From A’s statement
If A is truthful, th

In [20]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [21]:
print(together)

# Response from competitor 1

Let each person be either **T** (always truthful) or **F** (always deceptive).  
A, B, C, D each make exactly one statement, so:

- if someone is **T**, their statement is **true**;
- if someone is **F**, their statement is **false**.

We’ll translate each statement.

### Statements in logical form
1. **A**: “Exactly one of B and C is lying.”  
   \[
   A:\ (B \oplus C)
   \]
2. **B**: “D is truthful iff A is lying.”  
   “A is lying” means A is **F**. So:
   \[
   B:\ (D \text{ is T}) \Leftrightarrow (A \text{ is F})
   \]
3. **C**: “A and D are of opposite types.”  
   \[
   C:\ (A \neq D)
   \]
4. **D**: “B’s statement and C’s statement cannot both be true.”  
   “B’s statement is true” means B is truthful; similarly for C.  
   So:
   \[
   D:\ \neg(\text{(B is T)} \wedge \text{(C is T)})
   \]
   i.e.
   \[
   D:\ \neg(B_T \wedge C_T)
   \]
   equivalently: not both B and C are truthful.

---

## Solve by cases

### From A’s statement
If A is truthful

In [22]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [23]:
print(judge)

You are judging a competition between 5 competitors.
Each model has been given this question:

You are given a closed-book puzzle with four suspects—A, B, C, and D—each of whom may be either truthful or deceptive, but each makes exactly one statement. Their statements are:

- A: “Exactly one of B and C is lying.”
- B: “D is truthful if and only if A is lying.”
- C: “A and D are of opposite types.”
- D: “B’s statement and C’s statement cannot both be true.”

Assume each suspect is either always truthful or always deceptive, and that each statement is interpreted in the standard logical sense. Determine for every suspect whether they are truthful or deceptive, and explain why the solution is unique.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Her

In [24]:
judge_messages = [{"role": "user", "content": judge}]

In [26]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5.4",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)

{"results":["2","4","1","5","3"]}


In [27]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gemini-2.5-flash
Rank 2: glm-5
Rank 3: gpt-5.4-nano
Rank 4: moonshotai/kimi-k2.5
Rank 5: anthropic/claude-sonnet-4.6


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>